# Collect the LeJEPA encoder dataset (OGBench cube-single) → Hugging Face

Stage A of the LeJEPA identifiability pipeline (`LEJEPA_RUN.md` §4). One collector,
`scripts/data/collect_cube_single_ou.py`, produces **two** datasets:

| dataset | what it is |
|---|---|
| `cube_single_ou_<profile>.lance` | the OU positive pairs the encoder trains on |
| `cube_single_ou_style_<profile>.lance` | the style probe — the same collector with ρ pushed to ≈1, so the two views of a pair share their content and differ **only** in style |

The probe is not optional instrumentation. `run_metrics.py` arms `style_dataset` by
default, and without it `delta` cannot be split into nonlinearity and style leakage —
the reported bound then charges all of it to nonlinearity
(`scripts/identifiability/config/metrics.yaml`).

Flow: config → apply → install → smoke + throughput → collect → **dataset statistics** → upload → handoff.

## Where the statistics live

This notebook holds the **dataset-side** statistics: the phase-3 audit
(`identifiability/collect.py::audit_dataset`) — the recorded marginal and the achieved
lag-1 autocorrelation, compared against what the manifest *declares*. They belong here,
before the upload, because they are the only check that the collection pipeline did not
quietly alter the process it claims to carry (through clipping, rejection, or a reshape
that scrambled a coordinate), and discovering that after a multi-GB upload is discovering
it too late.

The **encoder-side** statistics — the frozen identifiability metric suite — need a
checkpoint, so they live in `train_lejepa_ogbcubesingle.ipynb`.

> No physics is stepped anywhere in this notebook. Every frame is an independent OU draw
> pushed through `render_content`, which is what makes this affordable at all
> (~6 ms/frame against 263 ms/frame through `reset()`).

## 1. Config

In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/ lands directly here

# --- Hugging Face ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')                     # ← paste here if not a pod env var
HF_REPO_ID = '<your-hf-username-or-org>/lejepa-ogbench-cube-single-ou'   # ← edit me
HF_PRIVATE = True

# --- what to collect ---
# The stage-A exit-criterion profile: the 9 physical DOFs + cube.size (n = 10).
# `task_content` (n = 13) adds cube.color and is a separately-reported arm, not this run.
PROFILE = 'physical_content'

NUM_PAIRS = 200_000          # encoder set. 2x that many rendered frames.
STYLE_PAIRS = 20_000         # style probe
SMOKE_PAIRS = 200            # throughput probe, thrown away
SEED = 3072
IMAGE_SIZE = 224

# ρ ≈ 1 for the style probe. The sampler enforces the paper's ρ ∈ (0, 1) strictly, so
# exactly 1 is rejected; at 1e-8 below it the residual content drift is ~7e-4 in z units
# (sub-pixel once rendered) against ~2.0 for a real step at ρ = 0.9.
STYLE_RHO = 0.99999999

# Rendering is ~90% of runtime. `egl` needs a GPU; use `osmesa` on a CPU-only pod
# (software rasterisation — several times slower). The collector auto-selects if unset.
MUJOCO_GL = 'egl'

# 1 = one process, no merge. See §5 before raising it: `swm merge` decodes and
# re-encodes every JPEG, so a sharded collection costs a second lossy generation.
NUM_SHARDS = 1
CLEANUP_SHARDS = True        # delete the per-shard tables after a successful merge

# --- derived, don't edit ---
OU_NAME = f'ogbench/cube_single_ou_{PROFILE}.lance'
STYLE_NAME = f'ogbench/cube_single_ou_style_{PROFILE}.lance'
SMOKE_NAME = f'ogbench/cube_single_ou_smoke_{PROFILE}.lance'
OGBENCH_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench')

## 2. Apply config

In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['MUJOCO_GL'] = MUJOCO_GL
if MUJOCO_GL == 'osmesa':
    os.environ.setdefault('PYOPENGL_PLATFORM', 'osmesa')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

os.makedirs(OGBENCH_DIR, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it behaves the same after a kernel restart

print('cwd           =', os.getcwd())
print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])
print('MUJOCO_GL     =', os.environ['MUJOCO_GL'])
print('datasets ->', OGBENCH_DIR)
!df -h "$STABLEWM_HOME"

## 3. Install dependencies

`[train]` pulls the collector's stack; `huggingface_hub` does the upload.

In [ ]:
%pip install -q -e '.[train,format]' huggingface_hub

## 4. Smoke collection and throughput

200 pairs through the real entry point. This is the renderer check — if `MUJOCO_GL` is
wrong it fails here, in a minute, rather than after you have committed the full run — and
it measures the frames/s this pod actually gets, which is what the ETA below is built from.

The reference machine does ~165 frames/s at 224×224. A CPU-only pod on `osmesa` will be
well under that.

In [ ]:
import shutil
import subprocess
import time
from pathlib import Path


HYDRA_QUIET = ['hydra.run.dir=.', 'hydra.output_subdir=null']


def sh(cmd):
    """Run a command, stream its output live, raise on a non-zero exit."""
    print('$', ' '.join(cmd), flush=True)
    started = time.time()
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end='')
    elapsed = time.time() - started
    if proc.returncode != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}')
    print(f'\n[{elapsed / 60:.1f} min]')
    return elapsed


def collector_cmd(dataset_name, num_pairs, rho=None, shard=0, num_shards=1):
    cmd = [
        'python', 'scripts/data/collect_cube_single_ou.py',
        f'latents.profile={PROFILE}',
        f'dataset_name={dataset_name}',
        f'num_pairs={num_pairs}',
        f'seed={SEED}',
        f'env.image_size={IMAGE_SIZE}',
        f'shard={shard}',
        f'num_shards={num_shards}',
        *HYDRA_QUIET,
    ]
    if rho is not None:
        cmd.append(f'program_constants.rho={rho}')
    return cmd

In [ ]:
smoke_seconds = sh(collector_cmd(SMOKE_NAME, SMOKE_PAIRS))

fps = (2 * SMOKE_PAIRS) / smoke_seconds
print(f'\nthroughput      ~{fps:.0f} frames/s  ({fps / 2:.0f} pairs/s)')
print(f'ETA encoder set  {2 * NUM_PAIRS / fps / 60:.0f} min  ({NUM_PAIRS:,} pairs)')
print(f'ETA style probe  {2 * STYLE_PAIRS / fps / 60:.0f} min  ({STYLE_PAIRS:,} pairs)')
if fps < 40:
    print('\n⚠ Well below the reference 165 frames/s. On a GPU pod set MUJOCO_GL=egl '
          'and re-run cells 1-2; on CPU this is simply what osmesa costs.')

Look at what it rendered. Top row is view A of each pair, bottom row is view B — same content one OU step apart, independent style draws.

In [ ]:
import io

import lance
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def pair_frames(dataset_name, count=6, offset=0):
    """Decode the first `count` pairs of a collected dataset as (view_a, view_b)."""
    path = Path(OGBENCH_DIR) / Path(dataset_name).name
    table = lance.dataset(str(path)).to_table(
        columns=['step_idx', 'pixels'], limit=2 * count, offset=2 * offset
    )
    steps = table.column('step_idx').to_numpy(zero_copy_only=False).ravel()
    blobs = table.column('pixels').to_pylist()
    views = [[], []]
    for step, blob in zip(steps, blobs):
        views[int(step)].append(np.array(Image.open(io.BytesIO(blob))))
    return views


def show_pairs(dataset_name, count=6, title=''):
    views = pair_frames(dataset_name, count)
    fig, axes = plt.subplots(2, count, figsize=(1.7 * count, 3.6), dpi=110)
    for row, (label, frames) in enumerate(zip(('view A', 'view B'), views)):
        for col in range(count):
            ax = axes[row, col]
            ax.imshow(frames[col])
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if col == 0:
                ax.set_ylabel(label, fontsize=9, color='#52514e')
    fig.suptitle(title or dataset_name, fontsize=10, color='#0b0b0b')
    fig.tight_layout()
    plt.show()


show_pairs(SMOKE_NAME, title='smoke collection — OU pairs at rho = 0.9')

## 5. Collect the encoder dataset

Single process by default. The collector is single-threaded, so `NUM_SHARDS > 1` is the
only way to use more cores — but it is not free:

- `swm merge` reads each shard back through the loader and writes it out again, which
  **decodes and re-encodes every JPEG** (a second lossy generation at quality 95).
- `swm merge` does not merge manifests. The merged sidecar below is shard 0's, copied
  verbatim — so its `num_pairs` field reads the *per-shard* count, not the total. The
  fields that matter downstream (`config_hash`, `latents.n`, `ou`, `style_range`,
  `excluded_pinned`) are identical across shards, which is why copying is sound; the
  per-shard manifests are kept and uploaded alongside so the discrepancy is traceable.

At the ETA printed above, one process is usually the better trade.

In [ ]:
def collect(dataset_name, num_pairs, rho=None, num_shards=1, cleanup=True):
    """Collect one dataset, sharding and merging only if asked to."""
    name = Path(dataset_name)
    stem, namespace = name.stem, name.parent

    if num_shards == 1:
        return sh(collector_cmd(dataset_name, num_pairs, rho=rho))

    started = time.time()
    procs = []
    logs = []
    for i in range(num_shards):
        log = Path(STABLEWM_HOME) / 'logs' / f'collect_{stem}_shard{i}.log'
        log.parent.mkdir(parents=True, exist_ok=True)
        handle = open(log, 'w')
        cmd = collector_cmd(dataset_name, num_pairs, rho=rho, shard=i, num_shards=num_shards)
        print('$', ' '.join(cmd))
        procs.append((subprocess.Popen(cmd, stdout=handle, stderr=subprocess.STDOUT), handle))
        logs.append(log)

    failed = []
    for i, (proc, handle) in enumerate(procs):
        rc = proc.wait()
        handle.close()
        if rc != 0:
            failed.append((i, logs[i]))
    if failed:
        raise RuntimeError(f'shards failed: {failed} — see the logs')

    sources = [f'{namespace}/{stem}_shard{i}.lance' for i in range(num_shards)]
    sh(['swm', 'merge', *sources, '--output', f'{namespace}/{stem}', '--overwrite'])

    # `swm merge` does not carry the manifest across; downstream derives the sidecar
    # name from the *dataset* name, so shard 0's has to be copied into place.
    shutil.copyfile(
        Path(OGBENCH_DIR) / f'{stem}_shard0_manifest.json',
        Path(OGBENCH_DIR) / f'{stem}_manifest.json',
    )
    print(f'copied shard-0 manifest -> {stem}_manifest.json '
          f'(its num_pairs field is the per-shard count)')

    if cleanup:
        for i in range(num_shards):
            for suffix in ('.lance', '_episodes.lance'):
                shard_path = Path(OGBENCH_DIR) / f'{stem}_shard{i}{suffix}'
                if shard_path.exists():
                    shutil.rmtree(shard_path)
        print(f'removed {num_shards} shard tables')

    elapsed = time.time() - started
    print(f'\n[{elapsed / 60:.1f} min total]')
    return elapsed

In [ ]:
collect(OU_NAME, NUM_PAIRS, num_shards=NUM_SHARDS, cleanup=CLEANUP_SHARDS)

## 6. Collect the style probe

The same collector, the same profile, `program_constants.rho` pushed to ≈1. Content is
then held fixed across the pair and style is still redrawn per view, so the two frames
differ only in appearance. Deliberately **not** sharded — 20k pairs is minutes, and the
merge round-trip would be pure loss.

In [ ]:
collect(STYLE_NAME, STYLE_PAIRS, rho=STYLE_RHO, num_shards=1)

## 7. Dataset statistics — the phase-3 gate

`audit_dataset` compares what was **recorded** against what the manifest **declares**.
Two things it can catch that nothing downstream can:

- **the marginal**: `latent/z` is read back out of the simulator, never copied from what
  the sampler asked for. Where a value was clipped to its declared bounds, an IK solve
  did not converge, or a coupled joint did not track its driver, the recorded marginal
  drifts off the standard normal the theory assumes. The collector logs the clipped
  fraction; this is where you see what it cost.
- **the autocorrelation**: ρ is the single number the whole bound is written in terms of
  (`D = δ / (2ρ(1−ρ))`). If the achieved lag-1 correlation is not the declared one, every
  metric row computed against it is quoting the wrong ρ.

Tolerances below are deliberately loose. A small deviation is the readback doing its job,
not a defect — the numbers are printed so you can judge the size, and only a gross
disagreement fails.

In [ ]:
import json
from types import SimpleNamespace

from stable_worldmodel.identifiability import collect as ident_collect
from stable_worldmodel.identifiability.ou import OUSampler


def read_manifest(dataset_name):
    stem = Path(dataset_name).stem
    return json.loads((Path(OGBENCH_DIR) / f'{stem}_manifest.json').read_text())


def read_latents(dataset_name):
    """(z, z_next) for every pair in a collected dataset."""
    path = Path(OGBENCH_DIR) / Path(dataset_name).name
    table = lance.dataset(str(path)).to_table(columns=['step_idx', 'latent/z'])
    steps = table.column('step_idx').to_numpy(zero_copy_only=False).ravel().astype(int)
    column = table.column('latent/z').combine_chunks()
    width = len(column[0])
    values = column.flatten().to_numpy(zero_copy_only=False).reshape(len(column), width)
    return values[steps == 0], values[steps == 1]


def audit(dataset_name, expect_rho, label):
    manifest = read_manifest(dataset_name)
    z, z_next = read_latents(dataset_name)
    n = int(manifest['latents']['n'])

    sampler = OUSampler(n=n, rho=np.asarray(manifest['ou']['rho'], dtype=float),
                        seed=int(manifest['ou']['seed']))
    report = ident_collect.audit_dataset(SimpleNamespace(n=n), sampler, z, z_next)

    rows = lance.dataset(str(Path(OGBENCH_DIR) / Path(dataset_name).name)).count_rows()
    drift = float(np.abs(z - z_next).max())
    mean_dev = float(np.abs(np.asarray(report['marginal_mean'])).max())
    var_dev = float(report['marginal_var_max_dev'])
    rho_err = float(report['rho_max_abs_error'])

    print(f'=== {label} — {dataset_name}')
    print(f'  pairs                {len(z):,}   (rows {rows:,})')
    print(f'  profile / n          {manifest["profile"]} / {n}')
    print(f'  config_hash          {manifest["config_hash"]}')
    print(f'  declared rho (mean)  {manifest["ou"]["rho_mean"]}')
    print(f'  isotropy             satisfied={manifest["ou"]["isotropy"]["satisfied"]} '
          f'margin={manifest["ou"]["isotropy"]["margin"]:+.5f}')
    # `.get`: manifests written before the style/contrast fixes (LEJEPA_RUN.md §12
    # defects 7 and 12) carry none of these keys, and a stale dataset should report
    # that rather than crash the audit.
    print(f'  style resampled      {manifest.get("resample_style_within_pair", "n/a")}   '
          f'min_contrast={manifest.get("min_contrast", "n/a")} '
          f'(applies={manifest.get("contrast_floor_applies", "n/a")})')
    print(f'  achieved rho         min {min(report["rho_achieved"]):.4f}  '
          f'max {max(report["rho_achieved"]):.4f}  max|err| {rho_err:.4f}')
    print(f'  marginal mean        max|mean| {mean_dev:.4f}')
    print(f'  marginal var         max|var-1| {var_dev:.4f}')
    print(f'  max |z - z_next|     {drift:.3e}')

    # Two separate questions, and conflating them is how a probe collected at the wrong
    # rho passes: (1) does the manifest declare the rho we asked the collector for, and
    # (2) does the data actually realise the rho the manifest declares.
    checks = [
        ('rows == 2 x pairs', rows == 2 * len(z)),
        (f'manifest declares rho = {expect_rho}',
         abs(float(manifest['ou']['rho_mean']) - expect_rho) < 1e-6),
        ('achieved rho matches the declared rho (within 0.05)', rho_err < 0.05),
        ('marginal mean within 0.10 of 0', mean_dev < 0.10),
        ('marginal var within 0.20 of 1', var_dev < 0.20),
    ]
    if expect_rho > 0.999:
        # The same threshold run_metrics.py warns at: above 1e-2 the "style" probe is
        # also absorbing an OU step, and style_sensitivity would score the transition.
        checks.append(('content drift < 1e-2 (probe is style-only)', drift < 1e-2))
    else:
        checks.append(('content actually moves (drift > 0.1)', drift > 0.1))

    for name, ok in checks:
        print(f'  [{"PASS" if ok else "FAIL"}] {name}')
    print()
    return all(ok for _, ok in checks), report


ou_ok, ou_report = audit(OU_NAME, 0.9, 'encoder set')
style_ok, style_report = audit(STYLE_NAME, STYLE_RHO, 'style probe')

if not (ou_ok and style_ok):
    raise RuntimeError('dataset audit failed — do not upload; fix the collection first')
print('both datasets pass the phase-3 audit.')

Per-dimension view of the same audit. `cube.pos_xy[0]` and `cube.pos_z` are the two
coordinates the camera framing is known to starve (27 px and ~156 px of travel against
`pos_xy[1]`'s 96 px) — if anything is going to show clipping or a depressed achieved ρ,
it is those.

In [ ]:
# `describe()` records EVERY registered latent -- content, style and excluded -- in
# registry order. `z` is the concatenation of the *content* ones only, in that same
# order (`LatentRegistry.content` / `.slices()`), so filtering by role and expanding
# by `dim` reproduces z's coordinate order exactly.
latent_names = []
for entry in read_manifest(OU_NAME)['latents']['latents']:
    if entry['role'] != 'content':
        continue
    dim = int(entry['dim'])
    latent_names += [entry['name'] if dim == 1 else f'{entry["name"]}[{i}]' for i in range(dim)]

assert len(latent_names) == len(ou_report['rho_declared']), (
    f'{len(latent_names)} content dims named but z is '
    f'{len(ou_report["rho_declared"])} wide'
)

print(f'{"latent":24s} {"rho_decl":>9s} {"rho_ach":>9s} {"mean":>9s} {"var":>9s}')
for i, name in enumerate(latent_names):
    print(f'{name:24s} {ou_report["rho_declared"][i]:9.4f} '
          f'{ou_report["rho_achieved"][i]:9.4f} '
          f'{ou_report["marginal_mean"][i]:9.4f} {ou_report["marginal_var"][i]:9.4f}')

The visual counterpart: in the probe the geometry is frozen and only appearance moves. If the cube shifts between the two rows, the probe was collected at the wrong ρ.

In [ ]:
show_pairs(OU_NAME, title=f'encoder set — OU pairs at rho = 0.9  ({PROFILE})')
show_pairs(STYLE_NAME, title=f'style probe — content held fixed, style redrawn  ({PROFILE})')

## 8. Upload to Hugging Face

Both `.lance` tables, both `_episodes.lance` side tables and both `_manifest.json`
sidecars, uploaded with the layout they have on disk. That layout matters: every consumer
of a manifest derives its name from the *dataset* name and looks for it **next to** the
dataset, so the sidecars have to land beside the tables, not in a subfolder. The training
notebook downloads straight into `$STABLEWM_HOME/datasets/ogbench/` and everything
resolves.

The smoke dataset is excluded.

In [ ]:
from huggingface_hub import HfApi

OU_STEM = Path(OU_NAME).stem
STYLE_STEM = Path(STYLE_NAME).stem

allow_patterns = []
for stem in (OU_STEM, STYLE_STEM):
    allow_patterns += [
        f'{stem}.lance/*',
        f'{stem}_episodes.lance/*',
        f'{stem}_manifest.json',
    ]
    if NUM_SHARDS > 1:
        allow_patterns.append(f'{stem}_shard*_manifest.json')

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_REPO_ID, repo_type='dataset', private=HF_PRIVATE, exist_ok=True)
print('patterns:', allow_patterns)

In [ ]:
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    folder_path=OGBENCH_DIR,
    path_in_repo='.',
    allow_patterns=allow_patterns,
    commit_message=(
        f'LeJEPA cube-single OU pairs — profile={PROFILE}, n={read_manifest(OU_NAME)["latents"]["n"]}, '
        f'{NUM_PAIRS:,} pairs at rho=0.9 + {STYLE_PAIRS:,}-pair style probe at rho~1, '
        f'{IMAGE_SIZE}px, seed={SEED}, config_hash={read_manifest(OU_NAME)["config_hash"]}'
    ),
)

## 9. Verify and hand off

In [ ]:
info = api.repo_info(HF_REPO_ID, repo_type='dataset', files_metadata=True)
total = sum(f.size or 0 for f in info.siblings)
print(f'{len(info.siblings)} files, {total / 1e9:.2f} GB')

expected = {f'{OU_STEM}_manifest.json', f'{STYLE_STEM}_manifest.json'}
present = {f.rfilename for f in info.siblings}
missing = expected - present
print('manifests present:', not missing, '' if not missing else f'MISSING {missing}')

print('\n--- paste into train_lejepa_ogbcubesingle.ipynb, cell 1 ---')
print(f"HF_REPO_ID = '{HF_REPO_ID}'")
print(f"PROFILE    = '{PROFILE}'")
print(f"SEED       = {SEED}")
print(f'# n = {read_manifest(OU_NAME)["latents"]["n"]}, '
      f'config_hash = {read_manifest(OU_NAME)["config_hash"]}, '
      f'{NUM_PAIRS:,} pairs')

Optional: reclaim the smoke dataset's disk.

In [ ]:
for suffix in ('.lance', '_episodes.lance'):
    path = Path(OGBENCH_DIR) / f'{Path(SMOKE_NAME).stem}{suffix}'
    if path.exists():
        shutil.rmtree(path)
smoke_manifest = Path(OGBENCH_DIR) / f'{Path(SMOKE_NAME).stem}_manifest.json'
smoke_manifest.unlink(missing_ok=True)
print('smoke dataset removed')